# Gradient Descent coarse search on Fritz

This notebook uses only the sparse Numba implementation from `gd_sparce_numba.py`.

In [1]:
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from als import predict_pairs_numba_kernel
from evaluation import rmse_from_predictions
from gd_sparce_numba import gradient_descent_numba

In [2]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml-latest"
RESULTS_DIR = PROJECT_ROOT / "results" / "gd_coarse"
HISTORIES_DIR = RESULTS_DIR / "histories"
HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_ratings.csv")
user_map = pd.read_csv(PROCESSED_DIR / "user_map.csv")
movie_map = pd.read_csv(PROCESSED_DIR / "movie_map.csv")

n_users = len(user_map)
n_movies = len(movie_map)

train_fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

val_user_idx = val_df["user_idx"].to_numpy(dtype=np.int64)
val_movie_idx = val_df["movie_idx"].to_numpy(dtype=np.int64)
val_ratings = val_df["rating"].to_numpy(dtype=np.float64)

print("Train fit shape:", train_fit_df.shape)
print("Validation shape:", val_df.shape)
print("Number of users:", n_users)
print("Number of movies:", n_movies)

Train fit shape: (21652583, 6)
Validation shape: (5413146, 6)
Number of users: 330975
Number of movies: 83239


In [3]:
def gd_validation_metrics(U, V, global_mean):
    predictions = predict_pairs_numba_kernel(
        U,
        V,
        val_user_idx,
        val_movie_idx,
        global_mean,
    )
    predictions = np.clip(predictions, 0.5, 5.0)
    return {
        "val_rmse": rmse_from_predictions(val_ratings, predictions),
    }

## Coarse hyperparameter grid

In [4]:
gd_rank_values = [10, 20, 30]
gd_eta_values = [1e-4, 3e-4]
gd_reg_values = [0.1, 1.0]
gd_init_scale_values = [0.01]
gd_grid_max_iter = 25

# Compile Numba kernels before the timed search.
warmup_result = gradient_descent_numba(
    train_fit_df.iloc[:1_000],
    n_users=n_users,
    n_movies=n_movies,
    rank=gd_rank_values[0],
    eta=gd_eta_values[0],
    reg=gd_reg_values[0],
    n_iter=1,
    init_scale=gd_init_scale_values[0],
    random_state=42,
    objective_tol=None,
)
del warmup_result

gd_histories = []
gd_failed_configs = []
gd_total_configs = (
    len(gd_rank_values)
    * len(gd_eta_values)
    * len(gd_reg_values)
    * len(gd_init_scale_values)
)
gd_config_number = 0
gd_search_start = perf_counter()

for rank in gd_rank_values:
    for eta in gd_eta_values:
        for reg in gd_reg_values:
            for init_scale in gd_init_scale_values:
                gd_config_number += 1
                history_path = HISTORIES_DIR / (
                    f"rank-{rank}_eta-{eta:g}_reg-{reg:g}_init-{init_scale:g}.csv"
                )
                print(
                    f"[{gd_config_number}/{gd_total_configs}] "
                    f"rank={rank}, eta={eta}, reg={reg}, init_scale={init_scale}"
                )

                if history_path.exists():
                    print("  using saved history")
                    gd_histories.append(pd.read_csv(history_path))
                    continue

                config_start = perf_counter()
                try:
                    _, _, _, history_df = gradient_descent_numba(
                        train_fit_df,
                        n_users=n_users,
                        n_movies=n_movies,
                        rank=rank,
                        eta=eta,
                        reg=reg,
                        n_iter=gd_grid_max_iter,
                        init_scale=init_scale,
                        random_state=42,
                        metrics_fn=gd_validation_metrics,
                        objective_tol=None,
                    )
                except FloatingPointError as error:
                    gd_failed_configs.append({
                        "rank": rank,
                        "eta": eta,
                        "reg": reg,
                        "init_scale": init_scale,
                        "error": str(error),
                    })
                    print("  skipped after divergence:", error)
                    continue

                config_seconds = perf_counter() - config_start
                history_df["config_seconds"] = config_seconds
                history_df.to_csv(history_path, index=False)
                gd_histories.append(history_df)

if not gd_histories:
    raise RuntimeError("Every GD configuration diverged")

gd_tuning_results_df = pd.concat(gd_histories, ignore_index=True)
gd_config_columns = ["rank", "eta", "reg", "init_scale"]
gd_best_by_config = (
    gd_tuning_results_df.loc[
        gd_tuning_results_df
        .groupby(gd_config_columns)["val_rmse"]
        .idxmin()
    ]
    .sort_values("val_rmse")
    .reset_index(drop=True)
)
gd_tuning_results_df.to_csv(RESULTS_DIR / "all_histories.csv", index=False)
gd_best_by_config.to_csv(RESULTS_DIR / "best_by_config.csv", index=False)

print(f"Grid search time: {(perf_counter() - gd_search_start) / 60:.1f} min")
if gd_failed_configs:
    print("Diverged configurations:", len(gd_failed_configs))

[1/12] rank=10, eta=0.0001, reg=0.1, init_scale=0.01


[2/12] rank=10, eta=0.0001, reg=1.0, init_scale=0.01


[3/12] rank=10, eta=0.0003, reg=0.1, init_scale=0.01


[4/12] rank=10, eta=0.0003, reg=1.0, init_scale=0.01


[5/12] rank=20, eta=0.0001, reg=0.1, init_scale=0.01


...

[9/12] rank=30, eta=0.0001, reg=0.1, init_scale=0.01


[10/12] rank=30, eta=0.0001, reg=1.0, init_scale=0.01


[11/12] rank=30, eta=0.0003, reg=0.1, init_scale=0.01


[12/12] rank=30, eta=0.0003, reg=1.0, init_scale=0.01


Grid search time: 15.9 min


## Coarse-search summary

In [5]:
print("Coarse search completed.")
display(gd_best_by_config)

Coarse search completed.


,iteration,rank,eta,reg,init_scale,objective,train_rmse_raw,train_rmse,relative_objective_decrease,val_rmse,config_seconds
0,25,30,0.0003,0.1,0.01,1.218562e+07,1.060920,1.060920,1.865692e-03,1.060881,102.355940
1,25,30,0.0003,1.0,0.01,1.218699e+07,1.060952,1.060952,1.842506e-03,1.060910,102.440058
2,25,20,0.0003,0.1,0.01,1.220510e+07,1.061769,1.061769,1.350270e-03,1.061654,76.326682
3,25,20,0.0003,1.0,0.01,1.220606e+07,1.061792,1.061792,1.333322e-03,1.061675,76.241398
4,25,10,0.0003,0.1,0.01,1.222784e+07,1.062759,1.062759,7.446042e-04,1.062560,58.493366
5,25,10,0.0003,1.0,0.01,1.222835e+07,1.062771,1.062771,7.352043e-04,1.062572,58.406551
6,25,30,0.0001,0.1,0.01,1.225487e+07,1.063931,1.063931,2.385483e-06,1.063654,102.543836
7,25,30,0.0001,1.0,0.01,1.225543e+07,1.063931,1.063931,2.372626e-06,1.063654,102.509578
8,25,20,0.0001,0.1,0.01,1.225500e+07,1.063937,1.063937,1.647629e-06,1.063656,76.234249
9,25,20,0.0001,1.0,0.01,1.225537e+07,1.063937,1.063937,1.638709e-06,1.063656,76.335935
